# 05 — Batch Inference: Full Track Detection with Sliding Window

Runs CNN-LSTM across all TDMS files to detect and localise anomalies along the full 4.16 km track.

The **Sliding Window (SW) post-processing** step corrects point-to-point misclassifications — this is the key architectural contribution of the CNN-LSTM-SW model.

## 1. Setup

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nptdms import TdmsFile
from collections import Counter
from tensorflow.keras.models import load_model

model = load_model("results/CNN_LSTM_final.h5")
with open("results/label_classes.json") as f:
    classes = json.load(f)

label_mapping = {i: cls for i, cls in enumerate(classes)}
print(f"Model loaded. Classes: {label_mapping}")

WINDOW_SIZE = 1000   # proximity window for SW post-processing
TDMS_DIR    = "data/raw/"

## 2. Feature Extraction Helper

In [ ]:
def extract_features_from_df(df):
    """Extract T&FD features from a single TDMS DataFrame."""
    td_rows = []
    for col in df.columns:
        x = df[col].values
        mean_val = np.mean(x)
        std_val  = np.std(x) + 1e-10
        pearson  = (mean_val - np.median(x)) / std_val
        kurt     = np.mean((x - mean_val)**4) / (std_val**2)**2
        td_rows.append([mean_val, pearson, kurt])
    td = pd.DataFrame(td_rows, columns=["mean","skewness","kurtosis"])

    fd_rows = []
    for col in df.columns:
        x   = df[col].values
        mag = np.abs(np.fft.fft(x))[:len(x)//2]
        mid = len(mag)//4
        ber  = np.sum(mag[:mid]**2) / (np.sum(mag[mid:]**2) + 1e-10)
        mu, sigma = np.mean(mag), np.std(mag) + 1e-10
        spec_kurt = np.mean((mag - mu)**4) / sigma**4
        contrast  = float(np.max(mag) - np.min(mag))
        fd_rows.append([ber, spec_kurt, contrast])
    fd = pd.DataFrame(fd_rows, columns=["band_energy_ratio","spectral_kurtosis","spectral_contrast"])

    return pd.concat([td, fd], axis=1)

## 3. Sliding Window Correction

In [ ]:
def sliding_window_correction(predictions, window_size=5, threshold=0.6):
    """
    Majority-vote sliding window to correct CNN-LSTM point-to-point errors.
    Key contribution of CNN-LSTM-SW (Section 4, Elsevier GEITS 2024).
    """
    corrected = predictions.copy()
    half = window_size // 2
    n = len(predictions)
    for i in range(n):
        window = predictions[max(0, i-half):min(n, i+half+1)]
        top_label, top_count = Counter(window).most_common(1)[0]
        if top_count / len(window) >= threshold:
            corrected[i] = top_label
    return corrected

## 4. Full Track Inference Loop

In [ ]:
all_results = []
tdms_files = sorted([f for f in os.listdir(TDMS_DIR) if f.endswith(".tdms")])

if not tdms_files:
    print("No TDMS files found in data/raw/ — add your files to run.")
else:
    for fname in tdms_files:
        print(f"Processing: {fname}")
        tdms_file = TdmsFile(os.path.join(TDMS_DIR, fname))
        data_dict = {ch.name: ch.data for ch in tdms_file.groups()[0].channels()}
        df = pd.DataFrame(data_dict)

        features = extract_features_from_df(df)
        X_new = features.values.reshape(features.shape[0], features.shape[1], 1)

        preds_raw = np.argmax(model.predict(X_new, verbose=0), axis=1)
        preds_sw  = sliding_window_correction(preds_raw, window_size=WINDOW_SIZE)

        for idx, (raw, sw) in enumerate(zip(preds_raw, preds_sw)):
            all_results.append({"file": fname, "location_idx": idx,
                                 "pred_raw": label_mapping[raw],
                                 "pred_sw":  label_mapping[sw],
                                 "corrected": raw != sw})

    results_df = pd.DataFrame(all_results)
    print(f"Total: {len(results_df):,} | Corrections: {results_df['corrected'].sum():,} "
          f"({results_df['corrected'].mean()*100:.1f}%)")
    print(results_df["pred_sw"].value_counts())
    results_df.to_csv("results/metrics/full_track_detections.csv", index=False)

## 5. Detection Plot Along Track

In [ ]:
if all_results:
    file_results = results_df[results_df["file"] == results_df["file"].iloc[0]]
    label_colors = {"NC":"#2196F3","TP":"#4CAF50","AC1":"#FF9800","AC2":"#F44336"}

    plt.figure(figsize=(16, 3))
    for label, color in label_colors.items():
        mask = file_results["pred_sw"] == label
        plt.scatter(file_results.loc[mask, "location_idx"],
                    [1] * int(mask.sum()), c=color, s=2, label=label, alpha=0.7)

    plt.title(f"CNN-LSTM-SW Detection — {file_results['file'].iloc[0]}",
              fontsize=11, fontweight="bold")
    plt.xlabel("Location Index (m along track)")
    plt.yticks([])
    plt.legend(loc="upper right", markerscale=5)
    plt.tight_layout()
    plt.savefig("results/figures/track_detection_output.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Add TDMS files to data/raw/ to generate detection plots.")